In [ ]:
#%% md
# Модель котировок MOEX — вторая половина системы "новости -> цена"
#
# ЦЕЛЬ: объединить текстовый энкодер (event-таблица + эмбеддинги из text_encoder_final.ipynb)
# с ценовым временным рядом в ОДНУ обучаемую модель, которая предсказывает будущее движение
# цены акции на заранее подобранный горизонт.
#
# ВАЖНОЕ УТОЧНЕНИЕ ПО ДАННЫМ (обнаружено по вашему файлу `01_base_analytics_news.ipynb`):
# у исходного telegram_news_raw.csv ЕСТЬ РОДНЫЕ поля `moex_tickers` и `news_types`
# (списки, например moex_tickers=['UGLD'], news_types=['Macro']). В текстовом энкодере
# тикеры извлекались регуляркой, а класс длительности — своим rule-based/zero-shot
# классификатором, БЕЗ использования этих родных полей. Здесь это исправляем: подтягиваем
# `moex_tickers`/`news_types` напрямую из сырого CSV как более надёжный источник и
# обогащаем ими признаки (см. раздел 2). Текстовый энкодер отдельно трогать не будем —
# чтобы не расширять уже сданную задачу — но помечаем это как рекомендацию на будущее.
#
# АРХИТЕКТУРА (две "половины", соединённые в одну модель):
#   PriceEncoder  — GRU по окну последних торговых дней (доходности, объём, волатильность).
#   NewsEncoder   — взвешенное пуллирование эмбеддингов АКТУАЛЬНЫХ на день t новостей
#                   (вес = significance_score * decay_weight(half_life_hours, elapsed) * REL/100,
#                   то есть буквально то, что мы посчитали в текстовом энкодере) + скалярные
#                   агрегаты (значимость, новизна, доля stale-новостей, типы новостей).
#   Fusion head   — MLP над конкатенацией price-вектора и news-вектора -> (доходность, направление).
#
# ГОРИЗОНТ ПРЕДСКАЗАНИЯ выбирается НЕ вручную, а эмпирически (раздел 4): считаем
# Information Coefficient (корреляцию новостного сигнала с форвардной доходностью) для
# нескольких горизонтов-кандидатов и берём тот, где сигнал сильнее всего. Кандидаты
# [1, 2, 3, 5, 10, 20] торговых дней выбраны по трём основаниям:
#   1) наши собственные half-life классов длительности из текстового энкодера — от 12ч
#      (заявления политиков) до 720ч=30 дней (смена руководства), т.е. и короткие, и
#      длинные горизонты нужны как кандидаты;
#   2) литература по предсказуемости доходности на основе новостного сентимента — типичный
#      диапазон устойчивой предсказуемости 1-14 дней (напр., новостной сентимент Bloomberg
#      предсказывает доходность именно на горизонте 7-14 дней, твиттер-сентимент — только
#      однодневную), многие ML-модели на новостях используют 10-дневный горизонт;
#   3) 20 торговых дней (~месяц) — стандартное окно для "медленных" эффектов (аналог
#      post-earnings-announcement drift), верхняя граница кандидатов.
#
# ЧЕСТНОЕ ПРЕДУПРЕЖДЕНИЕ: это исследовательская/учебная модель. Она НЕ учитывает
# транзакционные издержки, проскальзывание, ликвидность и не является инвестиционной
# рекомендацией. Раздел с "бэктестом" в конце — иллюстрация метрики, не торговая стратегия.
#%%


In [ ]:
from pathlib import Path
import os
import re
import ast
import json as _json
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 160)

RAW_DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('price_model_results')
OUT_DIR.mkdir(exist_ok=True)
FINAL_DIR = Path('price_model_output')
FINAL_DIR.mkdir(exist_ok=True)

WINDOW_DAYS = 14          # то же временное окно, что и в текстовом энкодере (для согласованности)
LOOKBACK_DAYS = 20        # сколько последних торговых дней подаём в PriceEncoder
MIN_EVENTS_PER_TICKER = 15  # тикеры с меньшим числом новостных событий не берём в обучение
CANDIDATE_HORIZONS = [1, 2, 3, 5, 10, 20]  # торговых дней — обоснование в шапке ноутбука

def find_file(filename, search_root='/kaggle/input'):
    """Ищет файл по имени в дереве /kaggle/input — на случай, если путь датасета с
    выходом текстового энкодера отличается от захардкоженного ниже."""
    if not os.path.isdir(search_root):
        return None
    for dirpath, _, filenames in os.walk(search_root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None

# По умолчанию ожидаем, что выход text_encoder_final.ipynb подключен как отдельный
# Kaggle-датасет. Поменяйте путь, если назвали датасет иначе.
NEWS_FEATURES_PATH = find_file('news_event_features.csv') or '/kaggle/input/text-encoder-output/news_event_features.csv'
EMBEDDINGS_PATH = find_file('event_embeddings.npy') or '/kaggle/input/text-encoder-output/event_embeddings.npy'
print('news_event_features.csv ->', NEWS_FEATURES_PATH)
print('event_embeddings.npy ->', EMBEDDINGS_PATH)

In [ ]:
#%% md
## 1. Загрузка артефактов текстового энкодера
#%%
if not os.path.exists(NEWS_FEATURES_PATH) or not os.path.exists(EMBEDDINGS_PATH):
    raise FileNotFoundError(
        'Не найдены артефакты текстового энкодера. Прикрепите к этому ноутбуку датасет с '
        'выходом text_encoder_final.ipynb (news_event_features.csv + event_embeddings.npy) '
        'через "Add Input", либо поправьте NEWS_FEATURES_PATH/EMBEDDINGS_PATH выше.'
    )

news_event_features = pd.read_csv(NEWS_FEATURES_PATH, parse_dates=['event_dttm'])
news_event_features['embedding_row_id'] = news_event_features['embedding_row_id'].astype(np.int64)
event_embeddings = np.load(EMBEDDINGS_PATH)
print(f'Событий (строк событие x тикер): {len(news_event_features):,}')
print(f'Уникальных событий: {news_event_features.event_id.nunique():,}')
print(f'Уникальных тикеров: {news_event_features.ticker.nunique():,}')
print(f'Эмбеддинги: {event_embeddings.shape}')
news_event_features.head()

In [ ]:
#%% md
## 2. Обогащение родными полями `moex_tickers` / `news_types` из сырого CSV
# Что применяем: text_encoder_final.ipynb извлекал тикеры регуляркой и класс длительности —
#     своим классификатором, не зная о родных полях `moex_tickers`/`news_types` в исходном
#     CSV. Здесь пересобираем то же выравнивание строк (та же нормализация текста + тот же
#     фильтр непустых текстов + reset_index), что и в текстовом энкодере — тогда `_row_id`
#     совпадает с `embedding_row_id`, и можно подтянуть родные поля представителя события.
# Как применяем:
#   - Если у представителя события есть непустой moex_tickers — ЗАМЕНЯЕМ им набор тикеров
#     события (родные теги надёжнее регулярки). Тикерам без REL-скора (появившимся только
#     из moex_tickers) присваиваем DEFAULT_REL — регулярка их не видела, но автотегирование
#     сочло их релевантными.
#   - news_types разворачиваем в бинарные флаги (type_Macro, type_Micro, ...) — добавляются
#     как ещё один скалярный признак в NewsEncoder (раздел 8).
# Как оцениваем результат: сравниваем долю событий с тикером ДО/ПОСЛЕ обогащения и
#     печатаем итоговое распределение типов новостей.
#%%
def normalize_text(text):
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

def parse_list(value):
    if pd.isna(value) or value in ('[]', ''):
        return []
    if isinstance(value, list):
        return value
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

DEFAULT_REL = 90.0  # для тикеров, пришедших только из родного moex_tickers (регулярка их не находила)

raw = pd.read_csv(RAW_DATA_PATH, parse_dates=['post_dttm'])
raw['text_normalized'] = raw['data'].fillna('').map(normalize_text)
raw = raw[raw['text_normalized'].str.len().gt(0)].copy().reset_index(drop=True)
raw['_row_id'] = np.arange(len(raw))
raw['moex_tickers_native'] = raw.get('moex_tickers', pd.Series([[]] * len(raw))).map(parse_list)
raw['news_types_native'] = raw.get('news_types', pd.Series([[]] * len(raw))).map(parse_list)
row_lookup = raw.set_index('_row_id')[['moex_tickers_native', 'news_types_native']]
print(f'Строк в выровненном сыром CSV: {len(raw):,} (должно совпадать по порядку с event_embeddings.npy)')

before_ticker_coverage = news_event_features.groupby('event_id')['ticker'].nunique()
print(f'Событий с >=1 тикером ДО обогащения: {(before_ticker_coverage > 0).sum():,}')

In [ ]:
EVENT_LEVEL_COLS = ['event_id', 'event_dttm', 'source', 'significance_score',
                     'significance_raw_count', 'significance_source_count',
                     'novelty_gap_days', 'is_stale', 'duration_class', 'half_life_hours',
                     'embedding_row_id', 'text_preview']

event_meta = news_event_features.drop_duplicates('event_id').set_index('event_id')[EVENT_LEVEL_COLS[1:]]
existing_rel = {}
for eid, g in news_event_features.groupby('event_id'):
    existing_rel[eid] = dict(zip(g['ticker'], g['rel_score']))

rebuilt_rows = []
for event_id, meta in event_meta.iterrows():
    row_id = meta['embedding_row_id']
    native_tickers, native_types = [], []
    if row_id in row_lookup.index:
        native_tickers = row_lookup.loc[row_id, 'moex_tickers_native']
        native_types = row_lookup.loc[row_id, 'news_types_native']

    rel_map = existing_rel.get(event_id, {})
    final_tickers = native_tickers if native_tickers else list(rel_map.keys())

    for ticker in final_tickers:
        rel_score = rel_map.get(ticker, DEFAULT_REL)
        row = {'event_id': event_id, 'ticker': ticker, 'rel_score': rel_score,
               'news_types': native_types}
        row.update(meta.to_dict())
        rebuilt_rows.append(row)

news_event_features = pd.DataFrame(rebuilt_rows)

after_ticker_coverage = news_event_features.groupby('event_id')['ticker'].nunique()
print(f'Событий с >=1 тикером ПОСЛЕ обогащения: {(after_ticker_coverage > 0).sum():,} из {len(event_meta):,}')

all_types = sorted({t for types in news_event_features['news_types'] for t in types})
print(f'Найденные категории news_types: {all_types}')
for t in all_types:
    news_event_features[f'type_{t}'] = news_event_features['news_types'].map(lambda lst, t=t: int(t in lst))

type_share = news_event_features[[f'type_{t}' for t in all_types]].mean().sort_values(ascending=False)
print('Доля событий по типам:')
print(type_share)

news_event_features.to_csv(OUT_DIR / 'news_event_features_enriched.csv', index=False, encoding='utf-8-sig')
news_event_features.head()

In [ ]:
#%% md
## 3. Отбор тикеров и загрузка дневных котировок MOEX
# Как применяем: берём тикеры с числом событий >= MIN_EVENTS_PER_TICKER (иначе слишком
#     мало сигнала для обучения по конкретной бумаге), тянем дневную историю OHLCV с
#     публичного MOEX ISS (без ключа) с запасом в LOOKBACK_DAYS торговых дней ДО начала
#     новостного периода и с запасом в max(CANDIDATE_HORIZONS) дней ПОСЛЕ его конца.
#%%
ticker_event_counts = news_event_features['ticker'].value_counts()
ticker_universe = ticker_event_counts[ticker_event_counts >= MIN_EVENTS_PER_TICKER].index.tolist()
print(f'Тикеров с >= {MIN_EVENTS_PER_TICKER} событий: {len(ticker_universe)}')
print(ticker_universe)

def load_moex_security_history(secid, board='TQBR', start=None, end=None):
    base = f'https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/{board}/securities/{secid}.json'
    frames, start_pos = [], 0
    while True:
        params = {'iss.meta': 'off', 'start': start_pos}
        if start: params['from'] = start
        if end: params['till'] = end
        try:
            resp = requests.get(base, params=params, timeout=20)
            resp.raise_for_status()
        except Exception:
            return pd.DataFrame()
        payload = resp.json()['history']
        rows = payload['data']
        if not rows:
            break
        frames.append(pd.DataFrame(rows, columns=payload['columns']))
        start_pos += len(rows)
        if len(rows) < 100:
            break
    if not frames:
        return pd.DataFrame()
    hist = pd.concat(frames, ignore_index=True)
    hist['TRADEDATE'] = pd.to_datetime(hist['TRADEDATE'])
    keep_cols = [c for c in ['TRADEDATE', 'CLOSE', 'OPEN', 'HIGH', 'LOW', 'VOLUME'] if c in hist.columns]
    return hist[keep_cols].dropna(subset=['CLOSE']).drop_duplicates('TRADEDATE').sort_values('TRADEDATE').reset_index(drop=True)

news_start = news_event_features.event_dttm.min().date()
news_end = news_event_features.event_dttm.max().date()
fetch_start = (news_event_features.event_dttm.min() - pd.Timedelta(days=LOOKBACK_DAYS * 2 + 10)).date().isoformat()
fetch_end = (news_event_features.event_dttm.max() + pd.Timedelta(days=max(CANDIDATE_HORIZONS) * 2 + 10)).date().isoformat()
print(f'Новостной период: {news_start} .. {news_end}')
print(f'Запрашиваем котировки: {fetch_start} .. {fetch_end}')

price_history = {}
for ticker in tqdm(ticker_universe, desc='Загрузка котировок MOEX'):
    hist = load_moex_security_history(ticker, start=fetch_start, end=fetch_end)
    if len(hist) >= LOOKBACK_DAYS + max(CANDIDATE_HORIZONS) + 5:
        price_history[ticker] = hist
    else:
        print(f'  {ticker}: недостаточно котировок ({len(hist)} дней) — исключён из обучения')

print(f'Тикеров с достаточной историей котировок: {len(price_history)} из {len(ticker_universe)}')

In [ ]:
#%% md
## 4. Ценовые признаки для PriceEncoder
# Как применяем: намеренно берём признаки, естественно нормализованные САМИ ПО СЕБЕ (без
#     отдельной нормализации со статистикой обучающей выборки — меньше риск утечки):
#       log_return       — log(close_t / close_t-1), уже безразмерная величина;
#       range_pct        — (high-low)/close, тоже безразмерная;
#       volume_ratio      — volume_t / rolling_mean(volume, 20) — относительный объём;
#       realized_vol_5d  — скользящее std лог-доходности за 5 дней (краткосрочная волатильность).
#     Все признаки причинные (используют только прошлые и текущий день, никакого заглядывания
#     вперёд).
# Как оцениваем результат: печатаем базовую статистику по признакам, чтобы отловить
#     аномалии (например, тикеры с нулевым объёмом или дырами в котировках).
#%%
PRICE_FEATURE_COLS = ['log_return', 'range_pct', 'volume_ratio', 'realized_vol_5d']

def build_price_features(hist):
    h = hist.copy()
    h['log_return'] = np.log(h['CLOSE']).diff()
    if 'HIGH' in h.columns and 'LOW' in h.columns:
        h['range_pct'] = (h['HIGH'] - h['LOW']) / h['CLOSE']
    else:
        h['range_pct'] = 0.0
    if 'VOLUME' in h.columns:
        roll_vol = h['VOLUME'].rolling(20, min_periods=5).mean()
        h['volume_ratio'] = (h['VOLUME'] / roll_vol.replace(0, np.nan)).fillna(1.0)
    else:
        h['volume_ratio'] = 1.0
    h['realized_vol_5d'] = h['log_return'].rolling(5, min_periods=2).std()
    h[PRICE_FEATURE_COLS] = h[PRICE_FEATURE_COLS].fillna(0.0)
    return h

price_features = {ticker: build_price_features(hist) for ticker, hist in price_history.items()}

sanity = pd.concat([df[PRICE_FEATURE_COLS].assign(ticker=t) for t, df in price_features.items()])
display(sanity.groupby('ticker')[PRICE_FEATURE_COLS].describe().T.head(20))

In [ ]:
#%% md
## 5. Эмпирический подбор горизонта предсказания
# Что проверяем: для каждого кандидата H (в торговых днях) считаем Information Coefficient
#     (ранговую корреляцию Спирмена) между простым дневным новостным сигналом на тикер и
#     форвардной доходностью через H торговых дней. Простой сигнал на этом шаге —
#     сумма significance_score по событиям, ОПУБЛИКОВАННЫМ В ЭТОТ ДЕНЬ по этому тикеру
#     (полная decay-взвешенная версия появится в финальной модели, здесь же — быстрая и
#     прозрачная диагностика, не зависящая от ещё не выбранного HORIZON_DAYS).
# Как оцениваем результат: горизонт с максимальным |IC| побеждает — считаем, что на этом
#     горизонте новостной сигнал ОБЪЯСНЯЕТ движение цены сильнее всего. Если несколько
#     горизонтов близки — берём меньший (короче срок эксплуатации модели, меньше риск
#     накопленного рыночного шума за длинное окно).
#%%
daily_signal_rows = []
for ticker in price_features:
    ev = news_event_features[news_event_features.ticker == ticker]
    daily_sig = ev.assign(day=ev.event_dttm.dt.date).groupby('day')['significance_score'].sum()
    daily_signal_rows.append(daily_sig.rename(ticker))
daily_signal = pd.concat(daily_signal_rows, axis=1) if daily_signal_rows else pd.DataFrame()

def forward_return_series(hist, horizon):
    close = hist.set_index('TRADEDATE')['CLOSE']
    fwd = close.shift(-horizon) / close - 1
    fwd.index = fwd.index.date
    return fwd

from scipy.stats import spearmanr

ic_by_horizon = {}
for horizon in CANDIDATE_HORIZONS:
    pairs_signal, pairs_return = [], []
    for ticker in price_features:
        if ticker not in daily_signal.columns:
            continue
        sig = daily_signal[ticker].dropna()
        fwd = forward_return_series(price_features[ticker], horizon)
        merged = pd.concat([sig, fwd.reindex(sig.index)], axis=1, keys=['signal', 'fwd_return']).dropna()
        merged = merged[merged['signal'] > 0]  # интересуют именно дни с новостями
        pairs_signal.extend(merged['signal'].tolist())
        pairs_return.extend(merged['fwd_return'].tolist())
    if len(pairs_signal) > 20:
        ic, pval = spearmanr(pairs_signal, pairs_return)
    else:
        ic, pval = np.nan, np.nan
    ic_by_horizon[horizon] = {'ic': ic, 'p_value': pval, 'n_obs': len(pairs_signal)}
    print(f'Горизонт {horizon:>2} дн.: IC={ic:.4f} (p={pval:.4f}, n={len(pairs_signal)})')

ic_report = pd.DataFrame(ic_by_horizon).T
ic_report.index.name = 'horizon_days'
ic_report.to_csv(OUT_DIR / 'horizon_ic_report.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ic_report.index.astype(str), ic_report['ic'].abs(), color='steelblue')
ax.set_title('|Information Coefficient| новостного сигнала по горизонтам-кандидатам')
ax.set_xlabel('Горизонт, торговых дней')
ax.set_ylabel('|IC| (корреляция Спирмена)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'horizon_selection_ic.png', dpi=150)
plt.show()

In [ ]:
#%% md
### Итоговый выбор горизонта
#%%
valid_ic = ic_report.dropna(subset=['ic'])
if valid_ic.empty:
    HORIZON_DAYS = 5
    print(f'Не удалось посчитать IC ни для одного горизонта (мало данных) — беру запасной '
          f'вариант HORIZON_DAYS={HORIZON_DAYS} (согласуется с half-life класса '
          f'"отчетность_дивиденды" и типичным окном 5-10 дней в литературе).')
else:
    HORIZON_DAYS = int(valid_ic['ic'].abs().idxmax())
    best_ic = valid_ic.loc[HORIZON_DAYS, 'ic']
    print(f'Выбран горизонт: {HORIZON_DAYS} торговых дней (IC={best_ic:.4f}, '
          f'n={int(valid_ic.loc[HORIZON_DAYS, "n_obs"])})')
    print('Обоснование: среди кандидатов [1, 2, 3, 5, 10, 20] торговых дней этот горизонт '
          'даёт максимальную по модулю ранговую корреляцию новостного сигнала со следующей '
          'доходностью — то есть новости на нашем датасете сильнее всего объясняют движение '
          'цены именно на этом сроке. Дальше вся модель (окно цен, таргет, метрики) строится '
          'под этот горизонт.')

with open(OUT_DIR / 'selected_horizon.json', 'w') as f:
    _json.dump({'horizon_days': HORIZON_DAYS}, f)

In [ ]:
#%% md
## 6. Сборка обучающей выборки (тикер x торговый день)
# Как применяем: для КАЖДОГО (тикер, день t), где есть и LOOKBACK_DAYS истории цены ДО t,
#     и известна цена через HORIZON_DAYS ПОСЛЕ t, собираем один сэмпл:
#       price_window   — последние LOOKBACK_DAYS дней ценовых признаков (раздел 4);
#       news_embedding — взвешенное пуллирование эмбеддингов АКТИВНЫХ на конец дня t
#                        событий по этому тикеру, вес = significance_score *
#                        decay_weight(half_life_hours, часов_с_публикации) * (rel_score/100)
#                        — ровно комбинация значимости, длительности эффекта и релевантности
#                        из текстового энкодера, теперь используемая по назначению;
#       news_scalar    — агрегаты по активным событиям (суммарный вес, число событий, доля
#                        stale, средний REL, флаги типов новостей);
#       target         — форвардная доходность close(t+H)/close(t)-1 и её знак.
#     "Активными" считаются события не старше MAX_RELEVANT_HOURS (180 дней) — decay_weight
#     дальше сам по себе гасит вклад старых событий почти до нуля, порог нужен только для
#     скорости вычислений (не сканировать всю историю тикера на каждый день).
# Как оцениваем результат: печатаем итоговое число сэмплов, долю дней с активными
#     новостями и распределение таргета по train/val/test.
#%%
def decay_weight(significance_score, half_life_hours, elapsed_hours):
    return significance_score * np.power(0.5, elapsed_hours / np.maximum(half_life_hours, 1e-6))

TOP_K_EVENTS = 8
MAX_RELEVANT_HOURS = 24 * 180
EMBED_DIM = event_embeddings.shape[1]
TYPE_COLS = [c for c in news_event_features.columns if c.startswith('type_')]
NEWS_SCALAR_FEATURE_NAMES = ['total_weight_log', 'n_active_log', 'share_stale', 'mean_rel'] + TYPE_COLS
N_SCALAR_FEATURES = len(NEWS_SCALAR_FEATURE_NAMES)

events_by_ticker = {
    t: g.sort_values('event_dttm').reset_index(drop=True)
    for t, g in news_event_features.groupby('ticker') if t in price_features
}

def active_events_for_day(ticker, day_cutoff):
    ev = events_by_ticker.get(ticker)
    if ev is None or len(ev) == 0:
        return None
    times = ev['event_dttm'].values
    right = np.searchsorted(times, np.datetime64(day_cutoff), side='right')
    left_bound = np.datetime64(day_cutoff - pd.Timedelta(hours=MAX_RELEVANT_HOURS))
    left = np.searchsorted(times, left_bound, side='left')
    if right <= left:
        return None
    return ev.iloc[left:right]

In [ ]:
samples = []
price_windows = []
news_pooled_embeds = []
news_scalars = []

for ticker, hist in tqdm(price_features.items(), desc='Сборка сэмплов по тикерам'):
    hist = hist.reset_index(drop=True)
    n = len(hist)
    for i in range(LOOKBACK_DAYS - 1, n - HORIZON_DAYS):
        day_t = hist.loc[i, 'TRADEDATE']
        close_t = hist.loc[i, 'CLOSE']
        close_future = hist.loc[i + HORIZON_DAYS, 'CLOSE']
        if pd.isna(close_t) or pd.isna(close_future) or close_t == 0:
            continue
        fwd_return = close_future / close_t - 1

        window = hist.loc[i - LOOKBACK_DAYS + 1:i, PRICE_FEATURE_COLS].to_numpy(dtype=np.float32)
        if window.shape[0] != LOOKBACK_DAYS:
            continue

        day_cutoff = day_t + pd.Timedelta(hours=23, minutes=59)
        active = active_events_for_day(ticker, day_cutoff)

        if active is not None and len(active) > 0:
            elapsed_hours = (day_cutoff - active['event_dttm']).dt.total_seconds() / 3600
            weights = decay_weight(active['significance_score'].to_numpy(),
                                    active['half_life_hours'].to_numpy(),
                                    elapsed_hours.to_numpy()) * (active['rel_score'].to_numpy() / 100)
            top_n = min(TOP_K_EVENTS, len(active))
            top_idx = np.argsort(-weights)[:top_n]
            top_weights = weights[top_idx]
            wsum = top_weights.sum()
            if wsum > 0:
                norm_w = top_weights / wsum
                emb_rows = active['embedding_row_id'].to_numpy().astype(np.int64)[top_idx]
                pooled = (event_embeddings[emb_rows] * norm_w[:, None]).sum(axis=0)
            else:
                pooled = np.zeros(EMBED_DIM, dtype=np.float32)
            scalar = [
                np.log1p(weights.sum()),
                np.log1p(len(active)),
                float(active['is_stale'].mean()),
                float(active['rel_score'].mean()),
            ] + [float(active[c].max()) for c in TYPE_COLS]
        else:
            pooled = np.zeros(EMBED_DIM, dtype=np.float32)
            scalar = [0.0, 0.0, 0.0, 0.0] + [0.0] * len(TYPE_COLS)

        samples.append({
            'ticker': ticker, 'date': day_t, 'fwd_return': fwd_return,
            'direction': int(fwd_return > 0), 'has_news': active is not None and len(active) > 0,
        })
        price_windows.append(window)
        news_pooled_embeds.append(pooled.astype(np.float32))
        news_scalars.append(np.array(scalar, dtype=np.float32))

samples_df = pd.DataFrame(samples)
price_windows = np.stack(price_windows)
news_pooled_embeds = np.stack(news_pooled_embeds)
news_scalars = np.stack(news_scalars)

print(f'Итого сэмплов: {len(samples_df):,}')
print(f'Из них с активными новостями: {samples_df.has_news.mean():.1%}')
print(f'price_windows: {price_windows.shape}, news_embeds: {news_pooled_embeds.shape}, news_scalars: {news_scalars.shape}')
samples_df['fwd_return'].describe()

In [ ]:
#%% md
### Разбиение train / val / test — строго по времени (без утечки будущего в прошлое)
#%%
date_sorted = np.sort(samples_df['date'].unique())
train_cutoff = date_sorted[int(len(date_sorted) * 0.70)]
val_cutoff = date_sorted[int(len(date_sorted) * 0.85)]

def assign_split(d):
    if d <= train_cutoff:
        return 'train'
    if d <= val_cutoff:
        return 'val'
    return 'test'

samples_df['split'] = samples_df['date'].map(assign_split)
print(samples_df['split'].value_counts())
print(f'train до {train_cutoff.date()}, val до {val_cutoff.date()}, test после')

split_idx = {s: samples_df.index[samples_df.split == s].to_numpy() for s in ['train', 'val', 'test']}
for s, idx in split_idx.items():
    print(f'{s}: {len(idx):,} сэмплов, доля направления up={samples_df.loc[idx, "direction"].mean():.2%}')

In [ ]:
#%% md
## 7. PyTorch Dataset и архитектура совместной модели
# PriceEncoder — GRU по окну LOOKBACK_DAYS ценовых признаков -> вектор.
# NewsEncoder  — линейная проекция пуллированного эмбеддинга + MLP по скалярным признакам.
# JointModel   — конкатенация price+news векторов -> MLP -> (доходность, логит направления).
#     Параметр use_news=False отключает новостную ветку целиком — так получаем "чистую"
#     price-only модель для честного сравнения в разделе 9 (ablation).
#%%
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство для обучения:', DEVICE)

class TickerNewsDataset(Dataset):
    def __init__(self, indices, price_windows, news_embeds, news_scalars, targets_return, targets_direction):
        self.indices = indices
        self.price_windows = price_windows
        self.news_embeds = news_embeds
        self.news_scalars = news_scalars
        self.targets_return = targets_return
        self.targets_direction = targets_direction

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return (
            torch.from_numpy(self.price_windows[idx]),
            torch.from_numpy(self.news_embeds[idx]),
            torch.from_numpy(self.news_scalars[idx]),
            torch.tensor(self.targets_return[idx], dtype=torch.float32),
            torch.tensor(self.targets_direction[idx], dtype=torch.float32),
        )

targets_return = samples_df['fwd_return'].to_numpy(dtype=np.float32)
targets_direction = samples_df['direction'].to_numpy(dtype=np.float32)

datasets = {
    s: TickerNewsDataset(idx, price_windows, news_pooled_embeds, news_scalars, targets_return, targets_direction)
    for s, idx in split_idx.items()
}
loaders = {
    s: DataLoader(ds, batch_size=128, shuffle=(s == 'train'))
    for s, ds in datasets.items()
}
print({s: len(ds) for s, ds in datasets.items()})

In [ ]:
class PriceEncoder(nn.Module):
    def __init__(self, n_features, hidden_size=32):
        super().__init__()
        self.gru = nn.GRU(input_size=n_features, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, x):
        _, h_n = self.gru(x)
        return self.norm(h_n[-1])

class NewsEncoder(nn.Module):
    def __init__(self, embed_dim, n_scalar_features, proj_dim=32, scalar_dim=16):
        super().__init__()
        self.embed_proj = nn.Sequential(nn.Linear(embed_dim, proj_dim), nn.ReLU())
        self.scalar_proj = nn.Sequential(nn.Linear(n_scalar_features, scalar_dim), nn.ReLU())

    def forward(self, news_embed, news_scalar):
        return torch.cat([self.embed_proj(news_embed), self.scalar_proj(news_scalar)], dim=-1)

class JointModel(nn.Module):
    def __init__(self, n_price_features, embed_dim, n_scalar_features, use_news=True,
                 price_hidden=32, news_proj_dim=32, news_scalar_dim=16):
        super().__init__()
        self.use_news = use_news
        self.price_encoder = PriceEncoder(n_price_features, hidden_size=price_hidden)
        fusion_dim = price_hidden
        if use_news:
            self.news_encoder = NewsEncoder(embed_dim, n_scalar_features, news_proj_dim, news_scalar_dim)
            fusion_dim += news_proj_dim + news_scalar_dim
        self.trunk = nn.Sequential(nn.Linear(fusion_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.return_head = nn.Linear(32, 1)
        self.direction_head = nn.Linear(32, 1)

    def forward(self, price_x, news_embed=None, news_scalar=None):
        p = self.price_encoder(price_x)
        if self.use_news:
            n = self.news_encoder(news_embed, news_scalar)
            fused = torch.cat([p, n], dim=-1)
        else:
            fused = p
        h = self.trunk(fused)
        return self.return_head(h).squeeze(-1), self.direction_head(h).squeeze(-1)

In [ ]:
#%% md
## 8. Обучение: функция с early stopping (общая для обеих моделей)
# Лосс = MSE по доходности + DIRECTION_LOSS_WEIGHT * BCE по направлению — учим оба выхода
#     одновременно, направление служит вспомогательной задачей, стабилизирующей обучение.
# Как оцениваем результат в процессе обучения: валидационный MSE по доходности,
#     ранняя остановка при отсутствии улучшения EARLY_STOP_PATIENCE эпох подряд.
#%%
DIRECTION_LOSS_WEIGHT = 0.3
EARLY_STOP_PATIENCE = 5
MAX_EPOCHS = 40

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    mse = nn.MSELoss()
    bce = nn.BCEWithLogitsLoss()
    total_loss, n_batches = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for price_x, news_e, news_s, y_ret, y_dir in loader:
            price_x, news_e, news_s = price_x.to(DEVICE), news_e.to(DEVICE), news_s.to(DEVICE)
            y_ret, y_dir = y_ret.to(DEVICE), y_dir.to(DEVICE)
            pred_ret, pred_dir_logit = model(price_x, news_e, news_s)
            loss = mse(pred_ret, y_ret) + DIRECTION_LOSS_WEIGHT * bce(pred_dir_logit, y_dir)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)

def train_model(use_news, n_price_features, embed_dim, n_scalar_features, tag):
    model = JointModel(n_price_features, embed_dim, n_scalar_features, use_news=use_news).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    best_val, best_state, patience = float('inf'), None, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_epoch(model, loaders['train'], optimizer)
        val_loss = run_epoch(model, loaders['val'])
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
        if val_loss < best_val - 1e-6:
            best_val, best_state, patience = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience += 1
        if epoch % 5 == 0 or patience == 0:
            print(f'[{tag}] эпоха {epoch:>2}: train_loss={train_loss:.5f} val_loss={val_loss:.5f}'
                  f'{" *" if patience == 0 else ""}')
        if patience >= EARLY_STOP_PATIENCE:
            print(f'[{tag}] ранняя остановка на эпохе {epoch} (best val_loss={best_val:.5f})')
            break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
#%% md
## 9. Обучение полной модели (price + news) и price-only абляции
#%%
print('=== Полная модель (price + news) ===')
model_full, history_full = train_model(
    use_news=True, n_price_features=len(PRICE_FEATURE_COLS),
    embed_dim=EMBED_DIM, n_scalar_features=N_SCALAR_FEATURES, tag='full')

print('\n=== Price-only модель (абляция без новостей) ===')
model_price_only, history_price_only = train_model(
    use_news=False, n_price_features=len(PRICE_FEATURE_COLS),
    embed_dim=EMBED_DIM, n_scalar_features=N_SCALAR_FEATURES, tag='price-only')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_full['epoch'], history_full['val_loss'], label='full (price+news) val')
ax.plot(history_price_only['epoch'], history_price_only['val_loss'], label='price-only val')
ax.set_xlabel('Эпоха')
ax.set_ylabel('Валидационный loss')
ax.set_title('Кривые обучения: полная модель vs price-only абляция')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
#%% md
## 10. Наивные бейзлайны (для честной шкалы сравнения)
#   - zero: всегда предсказывает нулевую доходность (обычная точка отсчёта для returns);
#   - train_mean: всегда предсказывает средний таргет по train (учитывает общий дрейф рынка
#     за период, но не тикер- и не новость-специфичен).
#%%
train_mean_return = samples_df.loc[split_idx['train'], 'fwd_return'].mean()
train_mean_direction = samples_df.loc[split_idx['train'], 'direction'].mean()
print(f'train_mean_return = {train_mean_return:.5f}, доля up в train = {train_mean_direction:.2%}')

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds_ret, preds_dir, ys_ret, ys_dir = [], [], [], []
    for price_x, news_e, news_s, y_ret, y_dir in loader:
        price_x, news_e, news_s = price_x.to(DEVICE), news_e.to(DEVICE), news_s.to(DEVICE)
        pred_ret, pred_dir_logit = model(price_x, news_e, news_s)
        preds_ret.append(pred_ret.cpu().numpy())
        preds_dir.append(torch.sigmoid(pred_dir_logit).cpu().numpy())
        ys_ret.append(y_ret.numpy())
        ys_dir.append(y_dir.numpy())
    return (np.concatenate(preds_ret), np.concatenate(preds_dir),
            np.concatenate(ys_ret), np.concatenate(ys_dir))

pred_full_ret, pred_full_dir, y_test_ret, y_test_dir = predict(model_full, loaders['test'])
pred_price_ret, pred_price_dir, _, _ = predict(model_price_only, loaders['test'])
pred_zero_ret = np.zeros_like(y_test_ret)
pred_mean_ret = np.full_like(y_test_ret, train_mean_return)

In [ ]:
#%% md
## 11. Оценка на тестовой выборке (все модели)
# Метрики: MAE/RMSE по доходности, корреляция Пирсона предсказание-факт, Information
#     Coefficient (Спирмен) — стандартная в квант-анализе метрика ранговой предсказательной
#     силы, accuracy/AUC по направлению движения.
#%%
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score, accuracy_score

def evaluate(name, pred_ret, y_ret, pred_dir=None, y_dir=None):
    mae = np.mean(np.abs(pred_ret - y_ret))
    rmse = np.sqrt(np.mean((pred_ret - y_ret) ** 2))
    pearson = pearsonr(pred_ret, y_ret)[0] if np.std(pred_ret) > 1e-9 else np.nan
    ic = spearmanr(pred_ret, y_ret)[0] if np.std(pred_ret) > 1e-9 else np.nan
    row = {'model': name, 'MAE': mae, 'RMSE': rmse, 'pearson_corr': pearson, 'IC_spearman': ic}
    if pred_dir is not None:
        pred_label = (pred_dir > 0.5).astype(int)
        row['direction_accuracy'] = accuracy_score(y_dir, pred_label)
        try:
            row['direction_auc'] = roc_auc_score(y_dir, pred_dir)
        except ValueError:
            row['direction_auc'] = np.nan
    return row

results = [
    evaluate('full (price+news)', pred_full_ret, y_test_ret, pred_full_dir, y_test_dir),
    evaluate('price-only (абляция)', pred_price_ret, y_test_ret, pred_price_dir, y_test_dir),
    evaluate('naive: zero', pred_zero_ret, y_test_ret),
    evaluate('naive: train_mean', pred_mean_ret, y_test_ret),
]
evaluation_report = pd.DataFrame(results).set_index('model')
display(evaluation_report.round(5))
evaluation_report.to_csv(OUT_DIR / 'evaluation_report.csv', encoding='utf-8-sig')

print(f'\nГоризонт предсказания: {HORIZON_DAYS} торговых дней')
print(f'Тестовых сэмплов: {len(y_test_ret):,}')
uplift_ic = evaluation_report.loc["full (price+news)", "IC_spearman"] - evaluation_report.loc["price-only (абляция)", "IC_spearman"]
print(f'Прирост IC от добавления новостей: {uplift_ic:+.4f}')

In [ ]:
#%% md
## 12. Визуализация качества предсказаний
#%%
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_ret, pred_full_ret, alpha=0.4, s=15, label='full (price+news)')
axes[0].scatter(y_test_ret, pred_price_ret, alpha=0.3, s=15, label='price-only', color='gray')
lims = [min(y_test_ret.min(), pred_full_ret.min()), max(y_test_ret.max(), pred_full_ret.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1, label='идеальное совпадение')
axes[0].set_xlabel(f'Факт. доходность за {HORIZON_DAYS} дн.')
axes[0].set_ylabel('Предсказанная доходность')
axes[0].set_title('Предсказание vs факт (тест)')
axes[0].legend(fontsize=8)

metrics_to_plot = ['IC_spearman', 'direction_accuracy']
plot_df = evaluation_report[metrics_to_plot].dropna()
plot_df.plot(kind='bar', ax=axes[1])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('IC и точность направления по моделям')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(OUT_DIR / 'prediction_quality.png', dpi=150)
plt.show()

In [ ]:
#%% md
## 13. Иллюстративная кривая накопленной доходности сигнала
# ВНИМАНИЕ: это ИЛЛЮСТРАЦИЯ метрики (аналог net trading signal), а НЕ бэктест и НЕ
#     торговая стратегия — здесь нет транзакционных издержек, проскальзывания, лимитов
#     ликвидности, funding-costs и рисков одновременного открытия позиций по многим
#     тикерам. Использовать для принятия инвестиционных решений НЕЛЬЗЯ.
# Как применяем: на каждый тестовый сэмпл сигнал = +1, если pred_ret > 0, иначе -1 (или 0
#     при предсказании ровно 0); кумулятивно суммируем sign(pred) * факт. доходность.
#%%
test_dates = samples_df.loc[split_idx['test'], 'date'].to_numpy()
signal = np.sign(pred_full_ret)
signal_price_only = np.sign(pred_price_ret)

order = np.argsort(test_dates)
cum_full = np.cumsum(signal[order] * y_test_ret[order])
cum_price_only = np.cumsum(signal_price_only[order] * y_test_ret[order])
cum_buy_hold = np.cumsum(y_test_ret[order])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(test_dates[order], cum_full, label='сигнал полной модели (price+news)')
ax.plot(test_dates[order], cum_price_only, label='сигнал price-only модели')
ax.plot(test_dates[order], cum_buy_hold, label='пассивное удержание (buy&hold по сэмплам)', linestyle=':')
ax.set_title(f'Иллюстративная накопленная доходность сигнала, горизонт={HORIZON_DAYS} дн. '
             '(БЕЗ издержек — не бэктест)')
ax.set_xlabel('Дата')
ax.set_ylabel('Накопленная доходность (сумма по сэмплам)')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT_DIR / 'illustrative_signal_curve.png', dpi=150)
plt.show()

In [ ]:
#%% md
## 14. Сохранение артефактов
#%%
torch.save(model_full.state_dict(), FINAL_DIR / 'joint_model_full.pt')
torch.save(model_price_only.state_dict(), FINAL_DIR / 'joint_model_price_only.pt')

config = {
    'horizon_days': HORIZON_DAYS,
    'lookback_days': LOOKBACK_DAYS,
    'min_events_per_ticker': MIN_EVENTS_PER_TICKER,
    'ticker_universe': list(price_features.keys()),
    'price_feature_cols': PRICE_FEATURE_COLS,
    'news_scalar_feature_names': NEWS_SCALAR_FEATURE_NAMES,
    'embed_dim': EMBED_DIM,
    'top_k_events': TOP_K_EVENTS,
    'max_relevant_hours': MAX_RELEVANT_HOURS,
    'train_cutoff': str(train_cutoff.date()),
    'val_cutoff': str(val_cutoff.date()),
}
with open(FINAL_DIR / 'model_config.json', 'w', encoding='utf-8') as f:
    _json.dump(config, f, ensure_ascii=False, indent=2)

evaluation_report.to_csv(FINAL_DIR / 'evaluation_report.csv', encoding='utf-8-sig')
samples_df.to_csv(FINAL_DIR / 'samples_metadata.csv', index=False, encoding='utf-8-sig')

print('Сохранено в', FINAL_DIR.resolve())
for p in sorted(FINAL_DIR.iterdir()):
    print(' -', p.name)

In [ ]:
#%% md
## Итог
#%%
print(f'Горизонт предсказания выбран эмпирически: {HORIZON_DAYS} торговых дней.')
print(f'Тикеров в обучении: {len(price_features)}; сэмплов: {len(samples_df):,} '
      f'(train={len(split_idx["train"])}, val={len(split_idx["val"])}, test={len(split_idx["test"])}).')
print('\nСравнение на тесте (см. evaluation_report выше):')
display(evaluation_report.round(5))
print('\nЧто дальше:')
print(' - если прирост IC от новостей на вашем прогоне мал/шумный — возможные причины:')
print('   мало сэмплов на длинных горизонтах, TOP_K_EVENTS/MAX_RELEVANT_HOURS требуют')
print('   подбора, decay_weight по классам длительности стоит откалибровать эмпирически')
print('   (см. раздел 2.4 текстового энкодера) вместо экспертных half-life;')
print(' - имеет смысл: 1) обучить text_encoder_final.ipynb отдавать moex_tickers/news_types')
print('   по ВСЕМ постам кластера, а не только по представителю; 2) сделать per-sector')
print('   или per-ticker fine-tuning; 3) добавить честный бэктест с издержками и лимитами')
print('   ликвидности перед любым практическим использованием сигнала.')